In [1]:
import numpy as np
from numpy.linalg import inv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.datasets import make_spd_matrix

import numba
from numba import jit

%matplotlib inline

In [21]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, rtol=1e-05, atol=1e-08):
    return np.allclose(a, a.T, rtol=rtol, atol=atol)

def vech(A):
    return A[np.tril_indices(A.shape[0])]

@jit(nopython=True)
def calc_vk(X, K, N):
    first_summand = 0
    second_summand = 0
    third_summand = 0
    
    # first summation
    for i in range(K+1):
        for j in range(K+1):
            if i == j:
                continue
            else:
                x_i = np.expand_dims(X[i], 1)
                x_j = np.expand_dims(X[j], 1)
                mult_one = vech(x_i.dot(x_i.T))
                mult_two = vech(x_j.dot(x_j.T))
                first_summand += mult_one*mult_two
    first_summand *= (1/(K*(K-1)))
    
    # second summation
    for i in range(N-1, K, -1):
        for j in range(N-1, K, -1):
            if i == j:
                continue
            else:
                x_i = np.expand_dims(X[i], 1)
                x_j = np.expand_dims(X[j], 1)
                mult_one = vech(x_i.dot(x_i.T))
                mult_two = vech(x_j.dot(x_j.T))
                second_summand += mult_one*mult_two
    second_summand *= (1/((N-K)*(N-K-1)))
    
    # third summation
    for i in range(K+1):
        for j in range(N-1, K, -1):
            x_i = np.expand_dims(X[i], 1)
            x_j = np.expand_dims(X[j], 1)
            mult_one = vech(x_i.dot(x_i.T))
            mult_two = vech(x_j.dot(x_j.T))
            third_summand += mult_one*mult_two
    third_summand *= (2/(K*(N-K)))
    
    total = first_summand + second_summand - third_summand
    return total

@numba.jit(nopython=True, parallel=True, cache=True)
def calc_vk_cached(X_vech, K, N, dim):
    first_summand = np.zeros(dim,)
    second_summand = np.zeros(dim,)
    third_summand = np.zeros(dim,)
    
    # first summation
    for i in numba.prange(K):
        for j in numba.prange(K):
            if i == j:
                continue
            else:
                mult_one = X_vech[i]
                mult_two = X_vech[j]
                first_summand = first_summand + (mult_one*mult_two)
    first_summand = first_summand * (1/(K*(K-1)))
    
    # second summation
    for i in numba.prange(K, N):
        for j in numba.prange(K, N):
            if i == j:
                continue
            else:
                mult_one = X_vech[i]
                mult_two = X_vech[j]
                second_summand = second_summand + (mult_one*mult_two)
    second_summand = second_summand * (1/((N-K)*(N-K-1)))
    
    # third summation
    for i in numba.prange(K):
        for j in numba.prange(K, N):
            mult_one = X_vech[i]
            mult_two = X_vech[j]
            third_summand = third_summand + (mult_one*mult_two)
    third_summand = third_summand * (2/(K*(N-K)))
    
    total = first_summand + second_summand - third_summand
    return total

In [22]:
N = 1000
P = 50
covar_one = make_spd_matrix(P)
covar_two = make_spd_matrix(P)
data_one = np.random.multivariate_normal(np.zeros(P), covar_one, N//2)
data_two = np.random.multivariate_normal(np.zeros(P), covar_two, N//2)
data = np.concatenate((data_one, data_two), 0)
# zero center - even though zero mean for sampling
data_centered = data - data.mean(axis=0)
X = data_centered
X.shape

(1000, 50)

In [23]:
X_expand = np.expand_dims(X, 2)
X_transp = np.transpose(X_expand, axes=(0,2,1))
X_mult = np.matmul(X_expand, X_transp)
X_vech = np.array([vech(x) for x in X_mult])
X_vech.shape

(1000, 1275)

In [27]:
#V_2 = calc_vk(X, K=2, N=N)
V_10 = calc_vk_cached(X_vech, K=10, N=N, dim=X_vech.shape[1])

In [13]:
@numba.jit(nopython=True, cache=True)
def calc_vs(X_vech, N):
    vs = []
    for k in numba.prange(2, N-2):
        vs.append(calc_vk_cached(X_vech, K=k, N=N))
    return np.array(vs)

In [14]:
vs = calc_vs(X_vech, N=N)

In [16]:
vs = np.array(vs)
vs.shape

(996, 1)

In [17]:
vs[0]

array([0.])

In [28]:
V_10

array([0., 0., 0., ..., 0., 0., 0.])